In [ ]:
import pandas as pd

In [ ]:
pip install -r requirements.txt

In [ ]:
df=pd.read_csv('data/Naukri Jobs Data.csv')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df = df.dropna(subset=['required_skills', 'job_post'])

In [ ]:
df["required_skills"] = df["required_skills"].apply(
    lambda x: [
        s.strip().lower().replace("_", " ").replace("-", " ")
        for s in str(x).replace("\\n", "\n").split("\n")
        if s.strip()
    ]
)

X = df["required_skills"].apply(lambda x: " , ".join(x) if isinstance(x, list) else x)


In [ ]:
df["required_skills"][0]

In [ ]:
unique_titles = df["job_post"].nunique()
print(f"🔹 Total unique job titles: {unique_titles}")

In [ ]:
import re

def normalize_job_title(title: str) -> str:
    """
    Cleans up job titles:
    - Removes seniority (Senior, Jr, Lead, etc.)
    - Removes numeric levels (I, II, III, 1, 2, 3)
    - Normalizes spacing
    - Keeps main title text (like 'Software Engineer', 'Data Scientist')
    """
    title = str(title).lower().strip()

    # Remove common seniority prefixes/suffixes
    title = re.sub(r'\b(senior|sr\.?|lead|principal|junior|jr\.?|entry[-\s]*level|intern|apprentice)\b', '', title)

    # Remove level indicators (I, II, III, 1, 2, 3, iv, v)
    title = re.sub(r'\b(i{1,3}|iv|v|vi{0,2}|[1-9])\b', '', title)

    # Remove extra hyphens, commas, parentheses
    title = re.sub(r'[-,()/]', ' ', title)

    # Collapse multiple spaces into one
    title = re.sub(r'\s+', ' ', title).strip()

    # Capitalize words for neatness
    title = title.title()

    return title


In [ ]:

y = df['job_post'].apply(normalize_job_title)

In [ ]:
unique_titles = y.nunique()
print(f"🔹 Total unique job titles: {unique_titles}")

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train = X_train.apply(lambda x: ' '.join(x) if isinstance(x, list) else x)
X_test = X_test.apply(lambda x: ' '.join(x) if isinstance(x, list) else x)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=8000,
    ngram_range=(1, 3),                # include unigrams, bigrams, trigrams
    token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z+\-_. ]+[a-zA-Z]\b'
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

feature_names = vectorizer.get_feature_names_out()

mean_tfidf = np.asarray(X_train_tfidf.mean(axis=0)).ravel()

top_n = 20
top_indices = mean_tfidf.argsort()[-top_n:][::-1]
top_features = feature_names[top_indices]
top_scores = mean_tfidf[top_indices]

plt.figure(figsize=(10, 6))
plt.barh(top_features[::-1], top_scores[::-1])  # reverse for visual ordering
plt.xlabel("Average TF-IDF Weight")
plt.title("Top 20 Most Important TF-IDF Features (Skills/Phrases)")
plt.tight_layout()
plt.show()


In [ ]:
X_train_tfidf.shape


In [ ]:
len(set(y_train))


In [ ]:
def simplify_job_title(title):
    title = title.lower().strip()

    # Data & AI roles
    if any(w in title for w in ["data scientist", "research scientist", "quant"]):
        return "data_scientist"
    if any(w in title for w in ["machine learning", "ml engineer", "ai engineer", "deep learning"]):
        return "ml_engineer"
    if any(w in title for w in ["data engineer", "etl", "pipeline", "big data"]):
        return "data_engineer"
    if "analyst" in title or "business intelligence" in title:
        return "data_analyst"

    # Software engineering roles
    if any(w in title for w in ["software engineer", "sde", "backend", "full stack", "full-stack"]):
        return "software_engineer"
    if any(w in title for w in ["frontend", "front-end", "ui developer", "react developer"]):
        return "frontend_engineer"
    if any(w in title for w in ["devops", "site reliability", "sre", "cloud engineer"]):
        return "devops_engineer"
    if any(w in title for w in ["mobile developer", "android", "ios", "flutter", "react native"]):
        return "mobile_engineer"

    # Product / Management roles
    if any(w in title for w in ["product manager", "pm", "program manager"]):
        return "product_manager"
    if any(w in title for w in ["project manager", "scrum master"]):
        return "project_manager"

    # Cybersecurity roles
    if any(w in title for w in ["security engineer", "cybersecurity", "infosec"]):
        return "security_engineer"

    # Cloud roles
    if any(w in title for w in ["aws", "azure", "gcp", "cloud architect"]):
        return "cloud_architect"

    # QA / Testing roles
    if any(w in title for w in ["qa", "test engineer", "automation engineer"]):
        return "qa_engineer"

    return "other"
y_train = y_train.apply(simplify_job_title)
y_test = y_test.apply(simplify_job_title)

In [ ]:
from sklearn.linear_model import SGDClassifier

model = SGDClassifier(
    loss="log_loss",      # logistic regression style
    max_iter=1000,
    tol=1e-3,
    n_jobs=-1,
    learning_rate="optimal",
    alpha=1e-4             # regularization strength
)
model.fit(X_train_tfidf, y_train)



In [ ]:
from sklearn.metrics import accuracy_score, f1_score

print(
    f"Accuracy: {accuracy_score(y_test, model.predict(X_test_tfidf)):.4f} | "
    f"Weighted F1: {f1_score(y_test, model.predict(X_test_tfidf), average='weighted'):.4f} | "
    f"Top-5 Acc: {np.mean([y in model.classes_[np.argsort(p)[-5:]] for y, p in zip(y_test, model.predict_proba(X_test_tfidf))]):.4f}"
)


In [ ]:
def get_top_skills_per_job(model, vectorizer, top_n=20):
    """Extract most important skill tokens for each job role."""
    feature_names = np.array(vectorizer.get_feature_names_out())
    job_skill_map = {}
    for i, job in enumerate(model.classes_):
        coefs = model.coef_[i]
        top_indices = np.argsort(coefs)[::-1][:top_n]
        job_skill_map[job] = list(feature_names[top_indices])
    return job_skill_map

job_top_skills = get_top_skills_per_job(model, vectorizer, top_n=20)

In [ ]:
def suggest_jobs(skills_text, top_n=5):
    """Predict top job matches with probabilities."""
    X_input = vectorizer.transform([skills_text])
    probs = model.predict_proba(X_input)[0]
    sorted_idx = np.argsort(probs)[::-1][:top_n]
    results = [(model.classes_[i], round(probs[i] * 100, 2)) for i in sorted_idx]
    return results


In [ ]:
import re

def clean_skill_text(text):
    text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)
    text = text.lower()
    text = text.replace("_", " ").replace("-", " ").replace("\n", " ")
    text = " ".join(text.split())
    return text

def get_job_suggestions(skills_input, top_n=5):
    if isinstance(skills_input, (list, tuple, set)):
        cleaned_skills = [clean_skill_text(s) for s in skills_input]
        skills_text = " , ".join(cleaned_skills)
    else:
        skills_text = clean_skill_text(str(skills_input))

    user_skills = set([s.strip() for s in skills_text.replace(",", " , ").split(" , ") if s.strip()])

    results = suggest_jobs(skills_text, top_n)

    print(f"\n💼 **Top {top_n} Job Matches:**\n")
    for job, score in results:
        print(f"🔹 {job} ({score}%)")

        ideal_skills = set([" ".join(skill.split()) for skill in job_top_skills.get(job, [])])

        learned = set()
        missing = set()

        for skill in ideal_skills:
            if any(skill in u or u in skill for u in user_skills):
                learned.add(skill)
            else:
                missing.add(skill)
        if ideal_skills:
            match_pct = round((len(learned) / len(ideal_skills)) * 100, 1)
            print(f"   ✅ Skill Match: {match_pct}% ({len(learned)}/{len(ideal_skills)} skills)")
            if learned:
                print(f"   🧠 You already know: {', '.join(sorted(learned))}")
            if missing:
                print(f"   🚀 Learn next: {', '.join(sorted(missing))}")
            else:
                print("   🎯 You already have all the top skills for this role!")
        else:
            print("   (No top skill data found for this role.)")

        print()
    return results


In [ ]:
get_job_suggestions([
    "Python","R","Java","SQL","Typescript","HTML5","CSS","Tableau","Power BI","Git",
    "React","Express.js","Node.js","Smart Contracts","PostgreSQL",
    "Object-Oriented Programming","Data Analysis","Data Visualization",
    "Machine Learning","Neural Networks","Deep Learning","Cloud Computing",
    "Algorithms","System architecture","Database Management Systems",
    "Data Structures","Software Engineering","Information Retrieval Systems",
    "Web Development","Quantum Computing","Feature Scaling","Collaboration",
    "Problem Solving","Analytical Thinking","Communication","Time Management",
    "Critical Thinking","Detail Orientation","Adaptability"
], top_n=5)

# 💬 EXPLANATION:
# The “data_engineer” label (and similar ones like “software_engineer”, “ml_engineer”, etc.)
# is not a single, narrow job title — it represents a *cluster* of closely related sub-roles
# that share many overlapping skills. For example:
#   data_engineer → { ETL Developer, Big Data Engineer, Data Pipeline Engineer, Hadoop Engineer }
#
# When the model predicts a "Job Match %" for this label:
#   → It reflects how close your *overall skill pattern* (as learned by the TF-IDF + classifier)
#     is to the *average skill distribution* across all roles inside that cluster.
#
# The "Skill Match %" on the other hand:
#   → Simply measures how many of the top skills for that cluster (e.g., Spark, ETL, Hadoop, Python)
#     appear directly in your own skill list.
#
# So if you ever see a lower Job Match % but a higher Skill Match %,
# that means your skills overlap strongly with the cluster,
# but your *overall skill pattern* (frequency and combination of skills)
# more closely resembles a different role.
#
# In short:
#   ▪ Job Match % → Statistical similarity to the cluster's average skill pattern.
#   ▪ Skill Match % → Literal overlap between your skills and the cluster's key skills.
